# Weak Label Baseline v1

This notebook checks the rating-derived weak labels and deterministic features. It is descriptive only: it does not train a sentiment model, use an LLM, or treat quality flags as ground truth.

**Leakage warning:** `weak_label` is derived directly from `rating`. Rating must not be used as a predictor in a future model trained on this label.

In [ ]:
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'config' / 'apps.yaml').exists():
            return candidate
    raise FileNotFoundError('Run the notebook from inside the project repository.')

PROJECT_ROOT = find_project_root()
DATASET_PATH = Path(os.environ.get(
    'DS_DATASET_PATH',
    PROJECT_ROOT / 'data' / 'processed' / 'review_features_v1.csv',
))
if not DATASET_PATH.is_file():
    raise FileNotFoundError(
        f'{DATASET_PATH} not found. Run scripts/build_ds_dataset.py first.'
    )

df = pd.read_csv(DATASET_PATH, keep_default_na=False)
label_order = ['negative', 'neutral', 'positive']
print(f'Loaded {len(df):,} rows from {DATASET_PATH}')

## 1. Schema, uniqueness, missing values, and types

In [ ]:
required = {
    'review_id', 'rating', 'app_name', 'vertical_name', 'published_at',
    'body_char_count', 'body_word_count', 'weak_label',
    'weak_label_needs_review', 'weak_label_noise_reasons',
}
missing_columns = required - set(df.columns)
assert not missing_columns, f'Missing columns: {sorted(missing_columns)}'
assert df['review_id'].is_unique, 'review_id must be unique'
assert set(df['weak_label']).issubset(set(label_order))

schema_check = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_count': df.isna().sum(),
    'unique_values': df.nunique(dropna=False),
})
display(schema_check)
print('Apps:', df['app_name'].nunique(), '| Verticals:', df['vertical_name'].nunique())

## 2. Rating and weak-label distributions

In [ ]:
rating_counts = df['rating'].value_counts().sort_index()
label_counts = df['weak_label'].value_counts().reindex(label_order, fill_value=0)
display(pd.DataFrame({'count': rating_counts, 'share': rating_counts / len(df)}))
display(pd.DataFrame({'count': label_counts, 'share': label_counts / len(df)}))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
rating_counts.plot.bar(ax=axes[0], title='Rating distribution', color='#4C78A8')
label_counts.plot.bar(ax=axes[1], title='Weak-label distribution', color='#F58518')
for ax in axes:
    ax.set_ylabel('Reviews')
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## 3. Differences by App and vertical

In [ ]:
app_labels = pd.crosstab(df['app_name'], df['weak_label'], normalize='index').reindex(columns=label_order, fill_value=0)
vertical_labels = pd.crosstab(df['vertical_name'], df['weak_label'], normalize='index').reindex(columns=label_order, fill_value=0)
app_ratings = df.groupby('app_name')['rating'].agg(['count', 'mean', 'median']).sort_values('mean')
vertical_ratings = df.groupby('vertical_name')['rating'].agg(['count', 'mean', 'median']).sort_values('mean')

display(app_ratings.join(app_labels.add_prefix('label_share_')))
display(vertical_ratings.join(vertical_labels.add_prefix('label_share_')))

app_labels.plot.barh(stacked=True, figsize=(9, 5), title='Weak-label share by App')
plt.xlabel('Share of App reviews')
plt.tight_layout()
plt.show()

## 4. Review-length behavior

In [ ]:
length_summary = df.groupby('weak_label')[['body_char_count', 'body_word_count']].describe()
display(length_summary)

plot_data = [
    df.loc[df['weak_label'].eq(label), 'body_char_count'].clip(upper=df['body_char_count'].quantile(0.99))
    for label in label_order
]
plt.figure(figsize=(8, 4))
plt.boxplot(plot_data, tick_labels=label_order, showfliers=False)
plt.title('Body character count by weak label (clipped at p99)')
plt.ylabel('Characters')
plt.show()

short_by_rating = pd.crosstab(
    df['rating'], df['quality_flag_too_short_review'], normalize='index'
).rename(columns={0: 'not_short_share', 1: 'short_share'})
display(short_by_rating)

## 5. Timestamp behavior

In [ ]:
timestamps = pd.to_datetime(df['published_at'], utc=True, errors='coerce')
assert timestamps.notna().all(), 'All published_at values should parse'
print('Observed range:', timestamps.min(), 'to', timestamps.max())

month_counts = df.groupby(['published_year', 'published_month']).size().rename('reviews')
weekday_counts = df['published_day_of_week'].value_counts()
display(month_counts.to_frame())
display(weekday_counts.to_frame('reviews'))

month_counts.plot.bar(figsize=(9, 3.5), title='Reviews by publication month', color='#54A24B')
plt.ylabel('Reviews')
plt.tight_layout()
plt.show()

## 6. Quality flags and issue signals

In [ ]:
quality_columns = [
    column for column in df.columns
    if column.startswith('quality_flag_') and column != 'quality_flag_count'
]
issue_columns = [
    column for column in df.columns
    if column.startswith('issue_') and column != 'issue_signal_count'
]
quality_counts = df[quality_columns].sum().sort_values(ascending=False)
issue_counts = df[issue_columns].sum().sort_values(ascending=False)
display(quality_counts.to_frame('reviews'))
display(issue_counts.to_frame('reviews'))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
quality_counts.plot.barh(ax=axes[0], title='Quality flags')
issue_counts.plot.barh(ax=axes[1], title='Issue signals')
plt.tight_layout()
plt.show()

issue_overlap = df[issue_columns].sum(axis=1).value_counts().sort_index()
display(issue_overlap.to_frame('reviews').rename_axis('issue_signals_per_review'))

## 7. Diagnostic checks

In [ ]:
diagnostics = {
    'largest_label_share': float(label_counts.max() / len(df)),
    'largest_app_share': float(df['app_name'].value_counts(normalize=True).max()),
    'reviews_needing_label_review': int(df['weak_label_needs_review'].sum()),
    'label_review_share': float(df['weak_label_needs_review'].mean()),
    'reviews_with_multiple_issue_signals': int((df['issue_signal_count'] > 1).sum()),
    'issue_signals_with_zero_matches': [c for c in issue_columns if int(df[c].sum()) == 0],
}
display(pd.Series(diagnostics, name='value').to_frame())

mismatch_by_label = pd.crosstab(
    df['weak_label'], df['quality_flag_rating_text_mismatch'], normalize='index'
).rename(columns={0: 'no_mismatch_share', 1: 'mismatch_share'})
mismatch_by_app = df.groupby('app_name')['quality_flag_rating_text_mismatch'].agg(['sum', 'mean']).sort_values('mean', ascending=False)
display(mismatch_by_label)
display(mismatch_by_app)

## 8. Anonymous examples requiring review

Reviewer identity is not present in this dataset. URLs and email-like strings are redacted below, source review IDs are omitted, and text is truncated.

In [ ]:
def redact_text(value, limit=180):
    text = re.sub(r'https?://\S+|www\.\S+', '[URL]', str(value))
    text = re.sub(r'\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b', '[EMAIL]', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text[:limit] + ('…' if len(text) > limit else '')

noise_examples = (
    df.loc[df['weak_label_needs_review'].eq(1), [
        'app_name', 'vertical_name', 'rating', 'weak_label',
        'weak_label_noise_reasons', 'body',
    ]]
    .sort_values(['weak_label_noise_reasons', 'app_name', 'rating'])
    .groupby('weak_label_noise_reasons', group_keys=False)
    .head(2)
    .head(12)
    .copy()
)
noise_examples['body_excerpt'] = noise_examples['body'].map(redact_text)
display(noise_examples.drop(columns='body'))

## 9. Interpretation and limitations

- Rating-derived labels are useful for exploration and an initial modeling target, but they are not manually verified sentiment ground truth.
- Mismatch flags and mixed-keyword signals identify review candidates; they do not prove that a rating is wrong.
- Sarcasm, rating mistakes, domain-specific meaning, emoji-only text, and mixed opinions are not reliably resolved by v1.
- The Apple RSS source exposes a changing recent window, not complete review history or a guaranteed production API.
- Before training a real classifier, manually annotate a stratified sample across labels, Apps, verticals, text lengths, and noise reasons.